# 02 — Train, calibrate, tune and test

    This small run follows the paper's decision workflow. It fits all six families
    on usable training attempts, selects candidates by validation Brier loss,
    calibrates on an independent split and chooses cutoffs on validation. The
    final test is generated only after those choices are fixed.

    Results are illustrative, not a replacement for the archived paper results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits

# The same notebook works from the repository root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/attendpro_paper2").exists())
thread_limit = threadpool_limits(limits=1)
pd.set_option("display.max_columns", 12)

In [ ]:
from attendpro_paper2 import evaluate, fit_models, pooled, predict, rule, simulate_attendpro, tune_rule

seed = 20263001
train = pooled(600, seed)
validation = pooled(300, seed + 100)
calibration = pooled(300, seed + 200)
display(pd.DataFrame({"split": ["training", "validation", "calibration"],
                      "attempts": [len(train), len(validation), len(calibration)]}))

## Candidate selection and independent calibration

    `fit_models` contains the six families and original candidate settings. It does
    not accept test data. Preprocessing is learned from training, not from all rows.
    Validation is reused for candidate selection and cutoff selection; calibration
    is separate. The calibrated score is not itself a pass/fail decision.

In [ ]:
models = fit_models(train, validation, calibration, seed=seed)
setting = tune_rule(validation, target=.01)
display(pd.DataFrame([{"method": name, "validation_Brier_usable": fitted["loss"],
                       "FAR_1pct_cutoff": fitted["thresholds"]["FAR_1pct"]}
                      for name, fitted in models.items()]))
print("Tuned rule (uncertainty weight, radius, similarity, liveness, consistency):", setting)

## Freeze choices, then generate fresh tests

    FAR and FRR include unusable attempts as rejections. The 1% constraint applies
    to validation; a small or shifted test may exceed it. Keep test data separate
    when experimenting with new models or thresholds.

In [ ]:
test = simulate_attendpro(5000, "nominal", seed + 1000)
decisions = {"Reference rule": rule(test), "Tuned hard rule": rule(test, setting)}
decisions.update({name: predict(test, fitted) for name, fitted in models.items()})
results = pd.DataFrame([dict(method=name, **evaluate(test.truth_valid_presence, accepted))
                        for name, accepted in decisions.items()])
display(results.assign(FAR_percent=100*results.FAR, FRR_percent=100*results.FRR)
               [["method", "FAR_percent", "FRR_percent", "n_valid", "n_invalid"]])

In [ ]:
ax = results.set_index("method")[["FAR", "FRR"]].mul(100).plot.bar(figsize=(10, 4))
ax.set(ylabel="Error (%)", title="Small nominal test: validation-selected operating point")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

destination = ROOT / "runs/notebook02"
destination.mkdir(parents=True, exist_ok=True)
results.to_csv(destination / "metrics.csv", index=False)

### Read the output carefully

    A lower FRR at a higher FAR is a trade-off, not dominance. A large apparent
    advantage from a small run may not persist across independent fits. Notebook
    04 reads the actual retained three-fit results. Notebook 03 examines the two
    shifts that motivate this paper: missing boundary coverage and facial dependence.